# September readings: anomalies

Salma's working notebook for the monthly check. It reads the month's file and the meter registry from `data/`, keeps the valid readings, and lists the readings that are far above what their meter usually shows.

In [1]:
import csv
from datetime import date
from statistics import median

with open("data/meters.csv", encoding="utf-8", newline="") as file:
    district_of = {row["meter_id"]: row["district"] for row in csv.DictReader(file)}
with open("data/readings_2026_09.csv", encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))
print(len(rows), "rows,", len(district_of), "meters")

17690 rows, 600 meters


In [3]:
readings = []          # (meter_id, date, m3) for every row that is kept
seen = set()
for row in rows:
    try:
        m3 = float(row["consumption_m3"])
        date.fromisoformat(row["date"])
    except ValueError:
        continue
    key = (row["meter_id"], row["date"])
    if not 0 <= m3 <= 200 or row["meter_id"] not in known or key in seen:
        continue
    seen.add(key)
    readings.append((row["meter_id"], row["date"], m3))
print(len(readings), "readings kept")

17561 readings kept


In [4]:
factor = 3.0       # a reading is an anomaly above this many times its meter's median

In [8]:
by_meter = {}
for meter_id, day, m3 in readings:
    by_meter.setdefault(meter_id, []).append(m3)
anomalies = [(meter_id, day, m3) for meter_id, day, m3 in readings
             if m3 > factor * median(by_meter[meter_id])]
print(len(anomalies), "anomalies")

27 anomalies


In [9]:
per_district = {}
for meter_id, day, m3 in anomalies:
    district = district_of[meter_id]
    per_district[district] = per_district.get(district, 0) + 1
for district in sorted(per_district):
    print(f"{district:15} {per_district[district]}")

Al-Faisaliyah   7
Al-Khuzama      5
Al-Murooj       4
Al-Naseem       3
Al-Rabwa        5
Al-Yasmin       3


## What if we were stricter?

Try a lower factor and look at the count again.

In [7]:
factor = 2.0